In [127]:
import numpy as np
import cvxpy as cvx

In [ ]:
def rsd_markowitz(Sigma, mu, r, reg, max_iter, random_P, p, distribution):
    n = Sigma.shape[0]

    A = np.concatenate( (mu, np.ones((1,n))) , axis = 0)
    b = np.array([[r],[1]])

    x = np.linalg.lstsq(A, b)[0]

    suboptimality = np.zeros( shape = (max_iter,1) )

    for iter in range(max_iter):
        if(random_P == True):
            p = np.random.randint(low = 1, high = n + 1)
        if(distribution == 'g'):
            # normal random
            S = np.random.normal( size = (n, p) )
        elif(distribution == 'u'):
            # uniform random
            S = np.random.uniform(low = 0, high = 1, size = (n, p) )
        else:
            # block sample
            ind = np.random.choice(n, p, replace = False)
            S = np.eye(n)
            S = S[:, ind]
        P_S = np.eye(p) - np.linalg.pinv( A @ S ) @ (A @ S)
        grad = Sigma @ x
        t = np.linalg.lstsq(P_S.T @ S.T @ Sigma @ S @ P_S + reg * np.eye(p), P_S.T @ S.T @ grad)[0]
        x = x - S @ P_S @ t

        suboptimality[iter] = x.T @ Sigma @ x

    x = cvx.Variable( (n,1) )
    cost = cvx.quad_form(x, Sigma, assume_PSD = True)
    constraints = [mu @ x == r, cvx.sum(x) == 1]
    prob = cvx.Problem(cvx.Minimize(cost), constraints)
    prob.solve()
    suboptimality = suboptimality - cost.value
    return suboptimality

In [129]:
n = 100
max_iter = 1000
admm_lambda = 1

In [130]:
rng = np.random.default_rng(4)
mu = rng.standard_normal(size=(1, n)) + 1
r = rng.standard_normal() + 1
Sigma = rng.standard_normal(size=(n, n))

Sigma = Sigma.T @ Sigma

In [131]:
np.linalg.cond(Sigma + 10*np.eye(n))

36.15718902411435

In [132]:
suboptimalities = rsd_markowitz(Sigma, mu, r, reg=100, max_iter=max_iter, random_P=False, p=0, distribution='g')

/var/folders/tw/zng4qsks70qffqd19hxk_1hr0000gn/T/ipykernel_89417/1904096678.py:7: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  x = np.linalg.lstsq(A, b)[0]


In [133]:
suboptimalities[max_iter-1]
suboptimalities[1] / suboptimalities[max_iter-1]

array([1.])